# D4 Transportation Workflow — GeoPandas

Transport network analysis using GeoPandas spatial operations.

In [ ]:
from __future__ import annotations
import json, os
from pathlib import Path
from urllib.error import URLError
from urllib.request import Request, urlopen
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

OUTPUT_DIR = Path.cwd() / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ALLOW_LIVE_API = os.getenv("GEOPROMPT_ALLOW_LIVE_API", "1") == "1"

def fetch_json(url, fallback):
    if not ALLOW_LIVE_API:
        return fallback
    try:
        req = Request(url, headers={"User-Agent": "geoprompt-notebook/2.0"})
        with urlopen(req, timeout=6) as r:
            return json.loads(r.read().decode("utf-8"))
    except (URLError, TimeoutError, ValueError):
        return fallback

def fetch_first_json(urls, validator, fallback):
    for url in urls:
        payload = fetch_json(url, None)
        if payload is not None and validator(payload):
            return payload, url, True
    return fallback, "fallback", False

import geopandas as gpd
from shapely.geometry import Point, Polygon, box
print(f"geopandas {gpd.__version__} ready")


## Section A: Pull Data Sources

In [ ]:
transport = {"features": [{"id": "fallback-transport"}]}
weather = {"properties": {"forecast": "fallback"}}
forecast = {"hourly": {"temperature_2m": [0.0]}}

transport, tr_src, tr_live = fetch_first_json(
    [
        "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_day.geojson",
        "https://api.github.com/repos/osm-search/Nominatim",
    ],
    lambda d: isinstance(d, dict) and bool(d.get("features") or d.get("id")),
    transport,
)
weather, wx_src, wx_live = fetch_first_json(
    [
        "https://api.weather.gov/points/40.75,-111.90",
        "https://api.weather.gov/points/34.05,-118.24",
    ],
    lambda d: isinstance(d, dict) and bool(d.get("properties", {}).get("forecast")),
    weather,
)
forecast, fc_src, fc_live = fetch_first_json(
    [
        "https://api.open-meteo.com/v1/forecast?latitude=40.75&longitude=-111.90&hourly=temperature_2m&forecast_days=1",
        "https://api.open-meteo.com/v1/forecast?latitude=34.05&longitude=-118.24&hourly=temperature_2m&forecast_days=1",
    ],
    lambda d: isinstance(d, dict) and len(d.get("hourly", {}).get("temperature_2m", [])) > 0,
    forecast,
)

transport_count = len(transport.get("features", [])) if isinstance(transport, dict) else 0
if transport_count == 0 and isinstance(transport, dict) and transport.get("id"):
    transport_count = 1
print(f"Transport records: {transport_count} | live={tr_live} | source={tr_src}")
print(f"NOAA forecast exists: {bool(weather.get('properties', {}).get('forecast'))} | live={wx_live} | source={wx_src}")
print(f"Open-Meteo hourly points: {len(forecast.get('hourly', {}).get('temperature_2m', []))} | live={fc_live} | source={fc_src}")


## Section B: Spatial Analysis (GeoPandas)

In [ ]:
nodes_data = {
    "node":        ["O",     "A",     "B",     "C",     "D",     "E",     "F"],
    "cost":        [0.0,     5.0,     9.0,     12.0,    14.0,    11.0,    3.0],
    "throughput":  [1500,    1200,    1100,    900,     800,     950,     1300],
    "node_type":   ["origin","inter","inter","dest","inter","inter","inter"],
}
lons = [-111.90, -111.87, -111.84, -111.78, -111.84, -111.81, -111.86]
lats = [  40.75,   40.75,   40.75,   40.75,   40.72,   40.72,   40.78]

gdf = gpd.GeoDataFrame(nodes_data, geometry=gpd.points_from_xy(lons, lats), crs="EPSG:4326")
print("Transport nodes:")
print(gdf[["node","cost","throughput"]].to_string(index=False))

# Incident data
incidents_gdf = gpd.GeoDataFrame(
    {"inc_id": ["I1","I2","I3"], "severity": [3, 1, 2]},
    geometry=gpd.points_from_xy([-111.88, -111.83, -111.79], [40.76, 40.73, 40.75]),
    crs="EPSG:4326",
)

# 1. Buffer: service zones around nodes (projected)
gdf_proj = gdf.to_crs("EPSG:3857")
service_buffers = gpd.GeoDataFrame(gdf[["node","cost"]], geometry=gdf_proj.buffer(3000).to_crs("EPSG:4326"))
print(f"\nService buffers (3km): {len(service_buffers)}")

# 2. Nearest join: assign incidents to nearest node
nearest = gpd.sjoin_nearest(incidents_gdf, gdf, how="left", distance_col="dist_to_node")
print("\nIncidents assigned to nearest node:")
print(nearest[["inc_id","severity","node","dist_to_node"]].to_string(index=False))

# 3. Spatial join: incidents within service buffers
in_buf = gpd.sjoin(incidents_gdf, service_buffers, how="left", predicate="within")
print("Incidents within 3km service zones:")
node_col = "node" if "node" in in_buf.columns else [c for c in in_buf.columns if c.startswith("node")][0]
print(in_buf[["inc_id", node_col]].dropna().to_string(index=False))

# 4. Dissolve by node type
dissolved = gdf.dissolve(by="node_type", aggfunc={"throughput": "sum", "cost": "mean"})
print("\nDissolved by node type:")
print(dissolved[["throughput","cost"]].to_string())

# 5. High-throughput nodes
high_tp = gdf[gdf["throughput"] > 1100]
print(f"\nHigh-throughput nodes (>1100): {list(high_tp['node'])}")

# 6. Bounding box: central corridor
central = gdf.cx[-111.90:-111.78, 40.73:40.77]
print(f"Nodes in central corridor: {list(central['node'])}")

# 7. Overlay: buffer intersections
buf_union = gpd.GeoDataFrame(gdf[["node"]], geometry=gdf_proj.buffer(5000).to_crs("EPSG:4326"))
overlaps = gpd.overlay(buf_union.iloc[:3], buf_union.iloc[1:4], how="intersection")
print(f"\nBuffer overlap areas: {len(overlaps)}")

gdf.to_file(str(OUTPUT_DIR / "d4-gpd-network.geojson"), driver="GeoJSON")
print("\nWrote d4-gpd-network.geojson")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
gdf.plot(ax=axes[0], column="cost", cmap="RdYlGn_r", markersize=160, legend=True,
         legend_kwds={"label": "Routing Cost"})
for _, row in gdf.iterrows():
    axes[0].annotate(row["node"], (row.geometry.x, row.geometry.y),
                     textcoords="offset points", xytext=(4,4), fontsize=9, fontweight="bold")
incidents_gdf.plot(ax=axes[0], color="red", markersize=100, marker="x", label="Incidents")
axes[0].set_title("Transport Network (x=incidents)")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

gdf_s = gdf.sort_values("cost")
axes[1].barh(gdf_s["node"], gdf_s["cost"], color="#0891b2")
axes[1].set_xlabel("Routing Cost"); axes[1].set_title("Node Reachability Costs")
axes[1].grid(True, axis="x", alpha=0.3)
plt.suptitle("D4 Transportation: GeoPandas Spatial Analysis", fontweight="bold")
plt.tight_layout(); plt.show()


## Section C: Scenario Comparison

In [ ]:
scenarios = {
    "no_action":    {"throughput": 1000, "avg_delay_min": 18.0, "cost_musd": 0.0},
    "signal_opt":   {"throughput": 1280, "avg_delay_min": 11.0, "cost_musd": 15.0},
    "new_corridor": {"throughput": 1600, "avg_delay_min":  7.0, "cost_musd": 85.0},
}
scenario_records = []
for name, vals in scenarios.items():
    score = round(vals["throughput"] / 1600 * 0.5 + (1.0 / vals["avg_delay_min"]) * 10 * 0.3
                  + (1.0 / max(vals["cost_musd"] + 1, 1)) * 20 * 0.2, 4)
    scenario_records.append({"scenario": name, **vals, "score": score})
scenario_records.sort(key=lambda r: -float(r["score"]))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
names = [r["scenario"] for r in scenario_records]
colors = ["#27ae60", "#e67e22", "#c0392b"]
axes[0].barh(names, [r["throughput"] for r in scenario_records], color=colors)
axes[0].set_xlabel("Throughput"); axes[0].set_title("Network Throughput"); axes[0].grid(True, axis="x", alpha=0.3)
axes[1].barh(names, [r["avg_delay_min"] for r in scenario_records], color=colors)
axes[1].set_xlabel("Avg Delay (min)"); axes[1].set_title("Average Delay"); axes[1].grid(True, axis="x", alpha=0.3)
axes[2].barh(names, [r["score"] for r in scenario_records], color=colors)
axes[2].set_xlabel("Composite Score"); axes[2].set_title("Scenario Score"); axes[2].grid(True, axis="x", alpha=0.3)
plt.suptitle("D4 Transportation (GeoPandas): Scenario Comparison", fontweight="bold")
plt.tight_layout(); plt.show()

(OUTPUT_DIR / "d4-gpd-complex.json").write_text(
    json.dumps({"scenario_ranking": scenario_records}, indent=2, default=str), encoding="utf-8"
)
print("Wrote d4-gpd-complex.json")
for r in scenario_records:
    print(f"  {r['scenario']}: score={r['score']}")
